# 04 | Mejoras al procesamiento de lenguaje (comentarios y títulos)
Propuestas de mejora del informe anterior, **implementadas y comparadas con la versión original**.

**Comentarios (Cuaderno 1):** se prueban tres cambios: (1) convertir emojis en tokens con una polaridad aproximada, (2) añadir n-gramas de caracteres para capturar jerga y errores de escritura, (3) ajustar el parámetro C. Las variantes se **eligen solo con validación cruzada por video dentro del entrenamiento**; la prueba se usa una sola vez al final.

**Títulos (Cuaderno 2):** se prueba (1) una lista de palabras vacías de dominio y (2) la elección de K por silueta en varios valores.

Cargue `comments.csv` y `videos-stats.csv` (nombres originales) y ejecute en orden. Semilla fija = 42.

In [1]:
import re, html, unicodedata, warnings
from pathlib import Path
import numpy as np, pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report
warnings.filterwarnings("ignore")
SEMILLA=42
for n in ["comments.csv","videos-stats.csv"]:
    if not Path(n).is_file():
        from google.colab import files; files.upload(); break
ETIQUETAS={0:"Negativo",1:"Neutral",2:"Positivo"}
STOP_ES=set("el la los las de del un una unos unas y o que en a al con por para es son se su sus lo como este esta estos estas mi mis tu tus me te le les ya muy mas pero si hay desde sobre entre".split())
STOP=(set(ENGLISH_STOP_WORDS)|STOP_ES)-{"no","not","nor","never","nunca","sin"}
PATRON=r"(?u)\b[^\W\d_][\w']*\b"
def normalizar(texto):
    t=unicodedata.normalize("NFKC",html.unescape(str(texto))).lower().replace("’","'")
    t=re.sub(r"https?://\S+|www\.\S+"," ",t); t=re.sub(r"<[^>]+>"," ",t)
    t=re.sub(r"\bwon't\b","will not",t); t=re.sub(r"\bcan't\b","can not",t); t=re.sub(r"n't\b"," not",t)
    return re.sub(r"\s+"," ",t).strip()
print("Listo.")

Listo.


## Parte A. Comentarios
### A1. Misma base y misma partición que el Cuaderno 1

In [2]:
c=pd.read_csv("comments.csv",encoding="utf-8-sig")
c=c.loc[:,~c.columns.str.match(r"^Unnamed")].drop_duplicates()
c=c[c["Comment"].notna()&c["Video ID"].notna()]
c["Sentiment"]=pd.to_numeric(c["Sentiment"],errors="coerce"); c=c[c["Sentiment"].isin(ETIQUETAS)].copy()
c["Sentiment"]=c["Sentiment"].astype(int); c["Comment"]=c["Comment"].astype(str); c["Video ID"]=c["Video ID"].astype(str).str.strip()
c["tn"]=c["Comment"].map(normalizar)
c=c[c["tn"].str.contains(PATRON,regex=True)]
conf=c.groupby("tn")["Sentiment"].nunique(); c=c[~c["tn"].isin(conf[conf>1].index)]
base=c.drop_duplicates("tn").reset_index(drop=True)
i_tr,i_te=next(StratifiedGroupKFold(5,shuffle=True,random_state=SEMILLA).split(base["Comment"],base["Sentiment"],groups=base["Video ID"]))
tr,te=base.iloc[i_tr].reset_index(drop=True),base.iloc[i_te].reset_index(drop=True)
print("Entrenamiento:",len(tr),"| Prueba:",len(te))

Entrenamiento: 14343 | Prueba: 3580


### A2. Mejoras de representación
- **Emojis:** los emojis positivos, negativos y otros se reemplazan por los tokens `emopos`, `emoneg` y `emojigen`. La lista de polaridad es una aproximación manual y revisable.
- **Caracteres:** n-gramas de 2 a 5 caracteres dentro de palabras (`char_wb`) ayudan con jerga y ortografía informal.

In [3]:
POS=set("😂🤣😍😊😀😁😃😄😆🥰😘❤💖💕💗💓💯👍👏🙌🔥🙏😎🤩✨🎉😻💙💚💛💜")
NEG=set("😡😠🤬😢😭😞😔💔👎🤮🤢😤😒🙄😩😫😱😨😰💀☹😟😖")
EMO=re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF\u2B50\u2764\uFE0F]")
def con_emojis(t):
    t=normalizar(t)
    t=re.sub("[\u2764]\uFE0F?"," emopos ",t)
    out=[]
    for ch in t:
        if ch in POS: out.append(" emopos ")
        elif ch in NEG: out.append(" emoneg ")
        elif EMO.match(ch): out.append(" emojigen ")
        elif ch=="\uFE0F": pass
        else: out.append(ch)
    return re.sub(r"\s+"," ","".join(out)).strip()
print(con_emojis("I love this 😂🔥 but not this 😡"))
print("Comentarios con algún emoji:",int(base["Comment"].map(lambda s:bool(EMO.search(s))).sum()),"de",len(base))

i love this emopos emopos but not this emoneg
Comentarios con algún emoji: 3664 de 17923


In [4]:
def vect_words(pre,max_f=20000):
    return TfidfVectorizer(preprocessor=pre,lowercase=False,token_pattern=PATRON+r"|[!?]",ngram_range=(1,2),min_df=2,max_features=max_f,sublinear_tf=True)
def vect_chars(pre):
    return TfidfVectorizer(preprocessor=pre,lowercase=False,analyzer="char_wb",ngram_range=(2,5),min_df=3,max_features=60000,sublinear_tf=True)
class Rep:
    def __init__(s,pre,chars): s.pre,s.chars=pre,chars
    def fit(s,x):
        s.w=vect_words(s.pre).fit(x); s.c=vect_chars(s.pre).fit(x) if s.chars else None; return s
    def transform(s,x):
        m=s.w.transform(x); return hstack([m,s.c.transform(x)]).tocsr() if s.chars else m
VARIANTES={"V0 original (palabras)":(normalizar,False),"V1 + emojis":(con_emojis,False),"V2 + emojis + caracteres":(con_emojis,True)}
print(list(VARIANTES))

['V0 original (palabras)', 'V1 + emojis', 'V2 + emojis + caracteres']


### A3. Elegir con validación cruzada por video (solo entrenamiento)

In [5]:
def cv_f1(pre,chars,C,k=3):
    f=[]
    for a,b in StratifiedGroupKFold(k,shuffle=True,random_state=SEMILLA).split(tr["Comment"],tr["Sentiment"],groups=tr["Video ID"]):
        r=Rep(pre,chars).fit(tr["Comment"].iloc[a]); m=LogisticRegression(max_iter=1000,class_weight="balanced",C=C,random_state=SEMILLA)
        m.fit(r.transform(tr["Comment"].iloc[a]),tr["Sentiment"].iloc[a])
        f.append(f1_score(tr["Sentiment"].iloc[b],m.predict(r.transform(tr["Comment"].iloc[b])),average="macro"))
    return float(np.mean(f)),float(np.std(f))
filas=[]
for nombre,(pre,ch) in VARIANTES.items():
    m,s=cv_f1(pre,ch,1.0); filas.append({"Variante":nombre,"C":1.0,"F1 macro CV (media)":m,"Desv.":s})
cv=pd.DataFrame(filas); display(cv.round(4))

,Variante,C,F1 macro CV (media),Desv.
0,V0 original (palabras),1.0,0.6675,0.0062
1,V1 + emojis,1.0,0.6709,0.0072
2,V2 + emojis + caracteres,1.0,0.6978,0.0058


In [6]:
mejor=cv.sort_values("F1 macro CV (media)",ascending=False).iloc[0]["Variante"]
pre,ch=VARIANTES[mejor]; filas=[]
for C in [0.5,1.0,2.0,4.0]:
    m,s=cv_f1(pre,ch,C); filas.append({"Variante":mejor,"C":C,"F1 macro CV (media)":m,"Desv.":s})
cv_c=pd.DataFrame(filas); display(cv_c.round(4))
C_MEJOR=float(cv_c.sort_values("F1 macro CV (media)",ascending=False).iloc[0]["C"])
print("Variante elegida:",mejor,"| C =",C_MEJOR)

,Variante,C,F1 macro CV (media),Desv.
0,V2 + emojis + caracteres,0.5,0.6895,0.0073
1,V2 + emojis + caracteres,1.0,0.6978,0.0058
2,V2 + emojis + caracteres,2.0,0.6996,0.0084
3,V2 + emojis + caracteres,4.0,0.7010,0.0114


Variante elegida: V2 + emojis + caracteres | C = 4.0


### A4. Evaluación final en la prueba (una sola vez)

In [7]:
def evalua(pre,ch,C):
    r=Rep(pre,ch).fit(tr["Comment"]); m=LogisticRegression(max_iter=1000,class_weight="balanced",C=C,random_state=SEMILLA)
    m.fit(r.transform(tr["Comment"]),tr["Sentiment"]); p=m.predict(r.transform(te["Comment"])); return p
p0=evalua(normalizar,False,1.0); p1=evalua(pre,ch,C_MEJOR)
def resumen(p): return {"Accuracy":accuracy_score(te["Sentiment"],p),"F1 macro":f1_score(te["Sentiment"],p,average="macro"),
  **{f"F1 {ETIQUETAS[k]}":f1_score(te["Sentiment"],p,labels=[k],average="macro") for k in ETIQUETAS},
  **{f"Recall {ETIQUETAS[k]}":(p[te["Sentiment"].to_numpy()==k]==k).mean() for k in ETIQUETAS}}
comp=pd.DataFrame({"Original":resumen(p0),"Mejorado":resumen(p1)}).T; comp["Δ vs original"]=np.nan
comp.loc["Mejorado","Δ vs original"]=comp.loc["Mejorado","F1 macro"]-comp.loc["Original","F1 macro"]
display(comp.round(4))
# Incertidumbre: bootstrap por video de la diferencia de F1 macro
rng=np.random.default_rng(SEMILLA); vids=te["Video ID"].unique(); grp={v:np.flatnonzero((te["Video ID"]==v).to_numpy()) for v in vids}; y=te["Sentiment"].to_numpy(); d=[]
for _ in range(300):
    ix=np.concatenate([grp[v] for v in rng.choice(vids,len(vids))]); d.append(f1_score(y[ix],p1[ix],average="macro")-f1_score(y[ix],p0[ix],average="macro"))
print("Diferencia de F1 macro (mejorado - original): %.4f | IC95%% bootstrap por video: [%.4f, %.4f]"%(np.mean(d),*np.percentile(d,[2.5,97.5])))

,Accuracy,F1 macro,F1 Negativo,F1 Neutral,F1 Positivo,Recall Negativo,Recall Neutral,Recall Positivo,Δ vs original
Original,0.7358,0.6607,0.5258,0.6049,0.8514,0.6358,0.6311,0.7992,NaN
Mejorado,0.7832,0.7039,0.5702,0.6580,0.8833,0.5905,0.6744,0.8678,0.0431


Diferencia de F1 macro (mejorado - original): 0.0439 | IC95% bootstrap por video: [0.0305, 0.0567]


## Parte B. Títulos
### B1. Palabras vacías de dominio y elección de K

In [8]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
v=pd.read_csv("videos-stats.csv",encoding="utf-8-sig"); v=v.loc[:,~v.columns.str.match(r"^Unnamed")]
v["Video ID"]=v["Video ID"].astype(str).str.strip(); v=v.drop_duplicates("Video ID",keep="first").reset_index(drop=True)
v["tn"]=v["Title"].fillna("").astype(str).map(html.unescape).map(normalizar)
DOMINIO={"video","videos","official","best","new","top","full","guide","watch","channel","live","episode","part","ft","feat","hd","tutorial","review","reviews","vs"}
def agrupa(stop,K,semilla=SEMILLA):
    vec=TfidfVectorizer(token_pattern=PATRON,lowercase=False,stop_words=sorted(stop),ngram_range=(1,2),min_df=2,max_df=.95,max_features=12000,sublinear_tf=True)
    M=vec.fit_transform(v["tn"]); ok=np.asarray(M.getnnz(axis=1)>0).ravel(); M=M[ok]
    X=Normalizer(copy=False).fit_transform(TruncatedSVD(50,random_state=semilla).fit_transform(M))
    g=KMeans(K,n_init=10,random_state=semilla).fit_predict(X)
    return silhouette_score(X,g,metric="cosine",sample_size=1000,random_state=semilla),g,vec.get_feature_names_out(),M,ok
filas=[]
for nombre,stop in [("Original",STOP),("Con palabras de dominio",STOP|DOMINIO)]:
    for K in [4,6,8,10]:
        s,*_=agrupa(stop,K); filas.append({"Stopwords":nombre,"K":K,"Silueta coseno":s})
tit=pd.DataFrame(filas); display(tit.pivot(index="K",columns="Stopwords",values="Silueta coseno").round(4))

Stopwords,Con palabras de dominio,Original
K,,
4,0.0690,0.0775
6,0.1027,0.1066
8,0.1370,0.1460
10,0.1679,0.1578


In [9]:
# Estabilidad: la silueta con 5 semillas para K=6, con y sin palabras de dominio
est=[]
for nombre,stop in [("Original",STOP),("Con palabras de dominio",STOP|DOMINIO)]:
    ss=[agrupa(stop,6,semilla=s)[0] for s in [1,7,21,42,99]]
    est.append({"Stopwords":nombre,"Media":np.mean(ss),"Desv.":np.std(ss),"Mín":min(ss),"Máx":max(ss)})
display(pd.DataFrame(est).round(4))

,Stopwords,Media,Desv.,Mín,Máx
0,Original,0.1068,0.0020,0.1031,0.1087
1,Con palabras de dominio,0.1078,0.0031,0.1027,0.1116


## Conclusiones
Complete con los números de arriba: ¿mejoró el F1 macro? ¿el intervalo de confianza incluye 0? ¿cambió la silueta más que la variación entre semillas? Una mejora dentro del ruido no debe presentarse como mejora real. Los embeddings multilingües quedan como trabajo futuro, porque requieren descargar un modelo externo.